# CardioIA — Fase 4 · Parte 2: Classificação de radiografias com CNN

**Tarefa:** classificar radiografias de tórax do NIH ChestX-ray14 em **`cardiomegalia`** ou **`normal`** (sem achados), usando o dataset preparado no notebook 01.

Comparamos duas abordagens:

| | Modelo A — CNN do zero | Modelo B — Transfer Learning |
|---|---|---|
| Arquitetura | 5 blocos convolucionais + GAP, cerca de 0,5 M parâmetros | **ResNet50** pré-treinada no ImageNet + cabeça densa |
| Entrada | 224×224×1, z-score com média e desvio do treino | 224×224×3 (cinza replicado), normalização do ImageNet |
| Treino | Todos os pesos, desde o início | Etapa 1: base congelada · Etapa 2: *fine-tuning* do bloco `conv5` |

As métricas são as vistas em aula (**acurácia, precisão, recall, F1 e matriz de confusão**), mais **AUC-ROC** e **especificidade**, que são comuns em diagnóstico médico. No final há um **protótipo interativo**: o usuário envia uma radiografia e recebe a classe, a probabilidade e um mapa de calor Grad-CAM mostrando onde o modelo olhou.

## 0. Configuração

In [ ]:
import os, sys, json, time, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/cardioia")
    DRIVE_DIR = Path("/content/drive/MyDrive/CardioIA-Fase4")
    DATA_DIR = BASE_DIR / "data" / "raiox_processado"
    if not (DATA_DIR / "indice.csv").exists():
        zipfile.ZipFile(DRIVE_DIR / "raiox_processado.zip").extractall(DATA_DIR)
    MODEL_DIR = DRIVE_DIR / "modelos"
else:
    BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_DIR = BASE_DIR / "data" / "raiox_processado"
    MODEL_DIR = BASE_DIR / "modelos"
FIG_DIR = BASE_DIR / "docs" / "prints" / "fase4"
for p in (MODEL_DIR, FIG_DIR):
    p.mkdir(parents=True, exist_ok=True)

RAPIDO = os.environ.get("CARDIOIA_RAPIDO") == "1"   # 1 época por etapa, só para testar o código
SEED, TAMANHO, BATCH = 42, 224, 32
EPOCAS_CNN = 1 if RAPIDO else 20
EPOCAS_TL_CONGELADO = 1 if RAPIDO else 30   # etapa rápida (features pré-calculadas)
EPOCAS_TL_FINETUNE = 1 if RAPIDO else 5
CLASSES = ["normal", "cardiomegalia"]   # índice 1 = classe positiva

In [ ]:
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix, classification_report,
                             roc_curve, precision_recall_curve, ConfusionMatrixDisplay)

keras.utils.set_random_seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "não (CPU)")

## 1. Carregamento do dataset pré-processado

In [ ]:
indice = pd.read_csv(DATA_DIR / "indice.csv")
norm = json.loads((DATA_DIR / "normalizacao.json").read_text(encoding="utf-8"))
indice["y"] = (indice.rotulo == "cardiomegalia").astype(int)
df = {s: indice[indice.split == s].reset_index(drop=True) for s in ("treino", "validacao", "teste")}

def carregar(caminho, y):
    img = tf.io.decode_png(tf.io.read_file(caminho), channels=1)
    return tf.ensure_shape(img, (TAMANHO, TAMANHO, 1)), y   # uint8 no cache (4x menos memória)

def criar_ds(d, treino=False):
    caminhos = [str(DATA_DIR / c) for c in d.caminho]
    ds = tf.data.Dataset.from_tensor_slices((caminhos, d.y.values))
    ds = ds.map(carregar, num_parallel_calls=tf.data.AUTOTUNE).cache()
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), tf.cast(y, tf.float32)))
    if treino:
        ds = ds.shuffle(len(d), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

ds_treino, ds_val, ds_teste = criar_ds(df["treino"], True), criar_ds(df["validacao"]), criar_ds(df["teste"])

# Pesos de classe: compensam o desbalanceamento 2:1 no cálculo da perda
n0, n1 = (df["treino"].y == 0).sum(), (df["treino"].y == 1).sum()
pesos_classe = {0: len(df["treino"]) / (2 * n0), 1: len(df["treino"]) / (2 * n1)}
pd.DataFrame({s: d.rotulo.value_counts() for s, d in df.items()}).T.assign(peso_classe_positiva=round(pesos_classe[1], 2))

## 2. Aumento de dados (*data augmentation*)

Aplicado **só durante o treino**, dentro do modelo, para reduzir overfitting:

* rotação de até ±7°, zoom de ±10% e translação de ±5%, que imitam diferenças de posicionamento do paciente;
* variação de contraste de ±10%, que imita diferenças entre aparelhos.

**Não usamos espelhamento horizontal.** O coração fica do lado esquerdo do tórax, e espelhar a imagem criaria uma anatomia irreal (dextrocardia). Isso poderia confundir justamente o achado que queremos detectar.

In [ ]:
def criar_aumento():
    return keras.Sequential([
        layers.RandomRotation(0.02, fill_mode="constant"),
        layers.RandomZoom(0.1, fill_mode="constant"),
        layers.RandomTranslation(0.05, 0.05, fill_mode="constant"),
        layers.RandomContrast(0.1),
    ], name="aumento")

aumento = criar_aumento()

exemplo = next(iter(ds_treino.unbatch().batch(1)))[0]
fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
axes[0].imshow(exemplo[0, ..., 0], cmap="gray", vmin=0, vmax=255); axes[0].set_title("original")
for a in axes[1:]:
    a.imshow(tf.clip_by_value(aumento(exemplo, training=True), 0, 255)[0, ..., 0], cmap="gray", vmin=0, vmax=255); a.set_title("aumentada")
for a in axes: a.axis("off")
plt.tight_layout(); plt.savefig(FIG_DIR / "02_aumento_dados.png", bbox_inches="tight"); plt.show()

## 3. Modelo A — CNN simples treinada do zero

Arquitetura enxuta e pensada para CPU: um *stem* com stride 2 reduz 224 → 112, depois vêm quatro blocos `Conv → BatchNorm → ReLU → MaxPool` com 64 → 128 → 128 → 256 filtros, *Global Average Pooling*, *Dropout* de 40% e uma saída sigmoide.

In [ ]:
def bloco(x, filtros, nome):
    x = layers.Conv2D(filtros, 3, padding="same", use_bias=False, name=f"{nome}_conv")(x)
    x = layers.BatchNormalization(name=f"{nome}_bn")(x)
    x = layers.ReLU(name=f"{nome}_relu")(x)
    return layers.MaxPooling2D(name=f"{nome}_pool")(x)

def construir_cnn():
    ent = keras.Input((TAMANHO, TAMANHO, 1), name="imagem")       # pixels 0–255
    x = criar_aumento()(ent)
    x = layers.Rescaling(1 / 255.0)(x)
    x = layers.Normalization(mean=norm["media"], variance=norm["desvio"] ** 2, name="zscore_treino")(x)
    x = layers.Conv2D(32, 5, strides=2, padding="same", use_bias=False, name="stem_conv")(x)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.ReLU(name="stem_relu")(x)
    for i, f in enumerate([64, 128, 128, 256], 1):
        x = bloco(x, f, f"bloco{i}")
    x = layers.Conv2D(256, 3, padding="same", activation="relu", name="ultima_conv")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    sai = layers.Dense(1, activation="sigmoid", name="prob_cardiomegalia")(x)
    return keras.Model(ent, sai, name="cnn_do_zero")

METRICAS = [keras.metrics.BinaryAccuracy(name="acuracia"), keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precisao"), keras.metrics.Recall(name="recall")]

cnn = construir_cnn()
cnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=METRICAS)
cnn.summary()

In [ ]:
def callbacks(nome, paciencia):
    return [
        keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=paciencia, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=max(2, paciencia // 2), verbose=1),
        keras.callbacks.CSVLogger(str(MODEL_DIR / f"historico_{nome}.csv"), append=True),
    ]

(MODEL_DIR / "historico_cnn_do_zero.csv").unlink(missing_ok=True)
t0 = time.time()
hist_cnn = cnn.fit(ds_treino, validation_data=ds_val, epochs=EPOCAS_CNN, class_weight=pesos_classe,
                   callbacks=callbacks("cnn_do_zero", 6), verbose=2)
tempo_cnn = (time.time() - t0) / 60
cnn.save_weights(MODEL_DIR / "cnn_do_zero.weights.h5")   # cópia de segurança imediata
print(f"Tempo de treino: {tempo_cnn:.1f} min")

## 4. Modelo B — Transfer Learning com ResNet50

A ResNet50 aprendeu filtros genéricos de bordas, texturas e formas com 1,2 milhão de imagens do ImageNet. Reaproveitamos esses filtros em duas etapas:

1. **Base congelada (extração de features):** com a base congelada, a saída da ResNet para cada imagem não muda entre as épocas. Por isso calculamos uma única vez o vetor de 2.048 features de cada radiografia e treinamos só a cabeça nova (Dropout → Dense) sobre esses vetores, com *learning rate* de 1e-3. Em CPU, isso troca ~15 min por época por alguns segundos. Nesta etapa não há aumento de dados.
2. ***Fine-tuning*:** descongelamos o último estágio (`conv5`) e treinamos o modelo completo, já com aumento de dados, usando *learning rate* 10 vezes menor (1e-4), para adaptar os filtros de alto nível à radiografia sem destruir o que já foi aprendido. As camadas de BatchNorm continuam congeladas, como recomenda a documentação do Keras.

A radiografia tem 1 canal, então replicamos o canal três vezes e aplicamos a mesma normalização usada no treino original da ResNet50 (subtração da média BGR do ImageNet).

In [ ]:
MEDIA_IMAGENET_BGR = [103.939, 116.779, 123.68]

def construir_resnet():
    ent = keras.Input((TAMANHO, TAMANHO, 1), name="imagem")       # pixels 0–255
    x = criar_aumento()(ent)
    x = layers.Concatenate(name="cinza_para_rgb")([x, x, x])
    x = layers.Normalization(mean=MEDIA_IMAGENET_BGR, variance=[1.0, 1.0, 1.0], name="norm_imagenet")(x)
    base = keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor=x)
    x = layers.GlobalAveragePooling2D(name="gap")(base.output)
    x = layers.Dropout(0.4, name="dropout_cabeca")(x)
    sai = layers.Dense(1, activation="sigmoid", name="prob_cardiomegalia")(x)
    modelo = keras.Model(ent, sai, name="resnet50_tl")
    # com input_tensor, base.layers inclui também as camadas de entrada; filtramos só as da ResNet
    return modelo, [l for l in base.layers if l.name.startswith(("conv", "pool"))]

resnet, camadas_base = construir_resnet()
for l in camadas_base:
    l.trainable = False
resnet.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=METRICAS)
print(f"Parâmetros treináveis na etapa 1: {sum(np.prod(w.shape) for w in resnet.trainable_weights):,} "
      f"de {resnet.count_params():,}")

In [ ]:
(MODEL_DIR / "historico_resnet50_tl.csv").unlink(missing_ok=True)
t0 = time.time()
# Etapa 1a: features da base congelada, calculadas uma única vez (sem embaralhar, para alinhar com os rótulos)
extrator = keras.Model(resnet.input, resnet.get_layer("gap").output)
feats = {s: extrator.predict(criar_ds(df[s]), verbose=0) for s in ("treino", "validacao")}
print(f"Features extraídas em {(time.time() - t0)/60:.1f} min:", {s: f.shape for s, f in feats.items()})

# Etapa 1b: cabeça treinada sobre as features (as camadas são as mesmas do modelo completo, pesos compartilhados)
ent_f = keras.Input((feats["treino"].shape[1],), name="features")
cabeca = keras.Model(ent_f, resnet.get_layer("prob_cardiomegalia")(resnet.get_layer("dropout_cabeca")(ent_f)), name="cabeca")
cabeca.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=METRICAS)
hist_tl1 = cabeca.fit(feats["treino"], df["treino"].y.values.astype("float32"), batch_size=BATCH,
                      validation_data=(feats["validacao"], df["validacao"].y.values.astype("float32")),
                      epochs=EPOCAS_TL_CONGELADO, class_weight=pesos_classe,
                      callbacks=callbacks("resnet50_tl", 5), verbose=2)
print(f"Etapa 1 concluída em {(time.time() - t0)/60:.1f} min")

In [ ]:
# Etapa 2: fine-tuning do estágio conv5 (BatchNorm segue congelado)
for l in camadas_base:
    l.trainable = l.name.startswith("conv5") and not isinstance(l, layers.BatchNormalization)
resnet.compile(optimizer=keras.optimizers.Adam(1e-4), loss="binary_crossentropy", metrics=METRICAS)
print(f"Parâmetros treináveis na etapa 2: {sum(np.prod(w.shape) for w in resnet.trainable_weights):,}")

hist_tl2 = resnet.fit(ds_treino, validation_data=ds_val, epochs=EPOCAS_TL_CONGELADO + EPOCAS_TL_FINETUNE,
                      initial_epoch=len(hist_tl1.epoch), class_weight=pesos_classe,
                      callbacks=callbacks("resnet50_tl", 3), verbose=2)
resnet.save_weights(MODEL_DIR / "resnet50_tl.weights.h5")   # cópia de segurança imediata
tempo_tl = (time.time() - t0) / 60
print(f"Tempo de treino total (etapas 1 + 2): {tempo_tl:.1f} min")

## 5. Curvas de treinamento

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for linha, (nome, titulo) in enumerate([("cnn_do_zero", "CNN do zero"), ("resnet50_tl", "ResNet50 (transfer learning)")]):
    h = pd.read_csv(MODEL_DIR / f"historico_{nome}.csv")
    for col, met in enumerate(["loss", "auc"]):
        ax = axes[linha, col]
        ax.plot(h.epoch + 1, h[met], "o-", label="treino", ms=3)
        ax.plot(h.epoch + 1, h[f"val_{met}"], "o-", label="validação", ms=3)
        if nome == "resnet50_tl" and len(hist_tl1.epoch) < len(h):
            ax.axvline(len(hist_tl1.epoch) + 0.5, color="gray", ls="--", lw=1)
            ax.text(len(hist_tl1.epoch) + 0.6, ax.get_ylim()[1], " fine-tuning", va="top", fontsize=8, color="gray")
        ax.set_title(f"{titulo} — {'perda' if met == 'loss' else 'AUC'}"); ax.set_xlabel("época"); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "02_curvas_treino.png", bbox_inches="tight"); plt.show()

## 6. Avaliação no conjunto de teste

**Escolha do limiar.** Um modelo sigmoide devolve uma probabilidade, e o limiar padrão de 0,5 nem sempre é o melhor quando as classes são desbalanceadas. Escolhemos, **no conjunto de validação**, o limiar que maximiza o F1. Depois aplicamos esse limiar fixo no teste, que até aqui não foi usado para nenhuma decisão.

In [ ]:
def prever(modelo, d):
    return modelo.predict(criar_ds(d), verbose=0).ravel()

def melhor_limiar(y, p):
    prec, rec, lim = precision_recall_curve(y, p)
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-9, None)
    return float(lim[np.argmax(f1[:-1])])

def metricas(y, p, limiar):
    yhat = (p >= limiar).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat).ravel()
    return {"limiar": round(limiar, 3), "acuracia": accuracy_score(y, yhat), "precisao": precision_score(y, yhat),
            "recall_sensibilidade": recall_score(y, yhat), "especificidade": tn / (tn + fp), "f1": f1_score(y, yhat),
            "auc_roc": roc_auc_score(y, p), "auc_pr": average_precision_score(y, p),
            "VP": int(tp), "FP": int(fp), "VN": int(tn), "FN": int(fn)}

modelos = {"CNN do zero": cnn, "ResNet50 (TL)": resnet}
resultados, preds_teste, preds_val, limiares = {}, {}, {}, {}
for nome, m in modelos.items():
    preds_val[nome] = prever(m, df["validacao"])
    lim = melhor_limiar(df["validacao"].y, preds_val[nome])
    p = prever(m, df["teste"])
    limiares[nome], preds_teste[nome] = lim, p
    resultados[nome] = metricas(df["teste"].y.values, p, lim)

tabela = pd.DataFrame(resultados).T
tabela.insert(0, "tempo_treino_min", [round(tempo_cnn, 1), round(tempo_tl, 1)])
tabela.insert(1, "parametros", [cnn.count_params(), resnet.count_params()])
tabela.style.format({c: "{:.3f}" for c in ["acuracia", "precisao", "recall_sensibilidade", "especificidade", "f1", "auc_roc", "auc_pr"]}) \
     .highlight_max(subset=["acuracia", "precisao", "recall_sensibilidade", "especificidade", "f1", "auc_roc", "auc_pr"], color="#d5f5e3")

In [ ]:
for nome, p in preds_teste.items():
    print(f"=== {nome} · limiar {limiares[nome]:.3f} ===")
    print(classification_report(df["teste"].y, (p >= limiares[nome]).astype(int), target_names=CLASSES, digits=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, (nome, p) in zip(axes, preds_teste.items()):
    ConfusionMatrixDisplay.from_predictions(df["teste"].y, (p >= limiares[nome]).astype(int), display_labels=CLASSES,
                                            cmap="Reds" if "zero" in nome else "Blues", ax=ax, colorbar=False)
    r = resultados[nome]
    ax.set_title(f"{nome}\nacurácia {r['acuracia']:.3f} · F1 {r['f1']:.3f} · recall {r['recall_sensibilidade']:.3f}", fontsize=10)
    ax.set_xlabel("previsto"); ax.set_ylabel("real")
plt.tight_layout(); plt.savefig(FIG_DIR / "02_matrizes_confusao.png", bbox_inches="tight"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
for nome, p in preds_teste.items():
    fpr, tpr, _ = roc_curve(df["teste"].y, p)
    axes[0].plot(fpr, tpr, label=f"{nome} (AUC {resultados[nome]['auc_roc']:.3f})")
    prec, rec, _ = precision_recall_curve(df["teste"].y, p)
    axes[1].plot(rec, prec, label=f"{nome} (AP {resultados[nome]['auc_pr']:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=.8); axes[0].set(xlabel="1 − especificidade (FPR)", ylabel="sensibilidade (TPR)", title="Curva ROC — teste")
axes[1].axhline(df["teste"].y.mean(), color="k", ls="--", lw=.8, label="aleatório")
axes[1].set(xlabel="recall", ylabel="precisão", title="Curva precisão × recall — teste")
for a in axes: a.legend(loc="lower right" if a is axes[0] else "lower left")
plt.tight_layout(); plt.savefig(FIG_DIR / "02_curvas_roc_pr.png", bbox_inches="tight"); plt.show()

In [ ]:
# Painel-resumo de métricas, usado como print da entrega
cols = ["acuracia", "precisao", "recall_sensibilidade", "especificidade", "f1", "auc_roc"]
rotulos = ["Acurácia", "Precisão", "Recall", "Especificidade", "F1-score", "AUC-ROC"]
x = np.arange(len(cols)); w = 0.38
fig, ax = plt.subplots(figsize=(10, 4))
for i, (nome, cor) in enumerate([("CNN do zero", "#e67e22"), ("ResNet50 (TL)", "#2471a3")]):
    vals = [resultados[nome][c] for c in cols]
    barras = ax.bar(x + (i - .5) * w, vals, w, label=nome, color=cor)
    ax.bar_label(barras, fmt="%.2f", fontsize=8, padding=2)
ax.set_xticks(x, rotulos); ax.set_ylim(0, 1.08); ax.legend(loc="lower right")
ax.set_title("Métricas no conjunto de teste (limiar escolhido na validação)")
plt.tight_layout(); plt.savefig(FIG_DIR / "02_comparativo_metricas.png", bbox_inches="tight"); plt.show()

## 7. Interpretabilidade com Grad-CAM

O Grad-CAM calcula o gradiente da saída em relação aos mapas da última camada convolucional e mostra **quais regiões da imagem mais pesaram na decisão**. Para um detector de cardiomegalia, esperamos que o calor fique na **silhueta cardíaca**. Se o calor aparecer em marcadores de texto, bordas da imagem ou eletrodos, é sinal de que o modelo aprendeu um atalho (*shortcut learning*).

In [ ]:
CAMADA_GRADCAM = {"CNN do zero": "ultima_conv", "ResNet50 (TL)": "conv5_block3_out"}

def gradcam(modelo, img, camada):
    # img: array (224, 224, 1) com pixels 0–255. Retorna (probabilidade, mapa 224x224 em 0–1).
    grad_model = keras.Model(modelo.inputs, [modelo.get_layer(camada).output, modelo.output])
    x = tf.convert_to_tensor(img[None].astype("float32"))
    with tf.GradientTape() as tape:
        mapas, pred = grad_model(x, training=False)
        alvo = pred[:, 0]
    grads = tape.gradient(alvo, mapas)
    pesos = tf.reduce_mean(grads, axis=(1, 2))
    cam = tf.nn.relu(tf.reduce_sum(mapas * pesos[:, None, None, :], axis=-1))[0]
    cam = cam / (tf.reduce_max(cam) + 1e-9)
    cam = tf.image.resize(cam[..., None], (TAMANHO, TAMANHO))[..., 0].numpy()
    return float(pred[0, 0]), cam

def ler_img(caminho):
    from PIL import Image
    return np.asarray(Image.open(caminho).convert("L").resize((TAMANHO, TAMANHO)), dtype=np.float32)[..., None]

teste = df["teste"].assign(p_tl=preds_teste["ResNet50 (TL)"])
casos = pd.concat([
    teste[(teste.y == 1) & (teste.p_tl >= limiares["ResNet50 (TL)"])].nlargest(3, "p_tl"),
    teste[(teste.y == 0) & (teste.p_tl < limiares["ResNet50 (TL)"])].nsmallest(2, "p_tl"),
    teste[(teste.y == 0) & (teste.p_tl >= limiares["ResNet50 (TL)"])].nlargest(1, "p_tl"),
])
fig, axes = plt.subplots(2, len(casos), figsize=(2.6 * len(casos), 5.6))
for j, r in enumerate(casos.itertuples()):
    img = ler_img(DATA_DIR / r.caminho)
    for i, nome in enumerate(["CNN do zero", "ResNet50 (TL)"]):
        p, cam = gradcam(modelos[nome], img, CAMADA_GRADCAM[nome])
        axes[i, j].imshow(img[..., 0], cmap="gray"); axes[i, j].imshow(cam, cmap="jet", alpha=.35)
        axes[i, j].set_title(f"{nome}\nreal: {r.rotulo} · p={p:.2f}", fontsize=8); axes[i, j].axis("off")
plt.suptitle("Grad-CAM — acertos (colunas 1–5) e um falso positivo (coluna 6)", y=1.0)
plt.tight_layout(); plt.savefig(FIG_DIR / "02_gradcam.png", bbox_inches="tight"); plt.show()

## 8. Salvando modelos e resultados

* `modelos/resnet50_tl.keras` e `modelos/cnn_do_zero.keras`: modelos usados pela API Flask e pelo app mobile, salvos sem o estado do otimizador.
* `modelos/config_modelos.json`: classes, limiares e métricas de teste.
* `modelos/predicoes_teste.csv` e `predicoes_validacao.csv`: probabilidades com idade, sexo e incidência, para a análise de fairness do notebook 03.

In [ ]:
def salvar_sem_otimizador(modelo, limpo, caminho):
    # Reconstrói a arquitetura sem compilar e copia só os pesos. Assim o arquivo não leva o estado
    # do otimizador Adam, o que reduz o tamanho em ~2x e mantém a ResNet50 abaixo do limite de 100 MB do GitHub.
    limpo.set_weights(modelo.get_weights())
    lote = next(iter(ds_teste))[0][:8]
    assert np.allclose(modelo(lote, training=False), limpo(lote, training=False), atol=1e-5)
    limpo.save(caminho)
    return caminho.stat().st_size / 1e6

tam = {"cnn_do_zero": salvar_sem_otimizador(cnn, construir_cnn(), MODEL_DIR / "cnn_do_zero.keras"),
       "resnet50_tl": salvar_sem_otimizador(resnet, construir_resnet()[0], MODEL_DIR / "resnet50_tl.keras")}
config = {
    "classes": CLASSES, "tamanho_entrada": TAMANHO, "entrada": "radiografia em escala de cinza, pixels 0-255",
    "modelo_padrao": "resnet50_tl",
    "modelos": {
        "resnet50_tl": {"arquivo": "resnet50_tl.keras", "nome": "ResNet50 (TL)", "camada_gradcam": "conv5_block3_out",
                        "limiar": limiares["ResNet50 (TL)"], "metricas_teste": resultados["ResNet50 (TL)"]},
        "cnn_do_zero": {"arquivo": "cnn_do_zero.keras", "nome": "CNN do zero", "camada_gradcam": "ultima_conv",
                        "limiar": limiares["CNN do zero"], "metricas_teste": resultados["CNN do zero"]},
    },
    "dataset": "NIH ChestX-ray14 — coorte cardiomegalia vs. sem achados (ver notebook 01)",
}
(MODEL_DIR / "config_modelos.json").write_text(json.dumps(config, indent=2, ensure_ascii=False, default=float), encoding="utf-8")

colunas = ["arquivo", "caminho", "paciente_id", "idade", "faixa_etaria", "sexo", "incidencia", "rotulo", "y"]
for split, preds in (("teste", preds_teste), ("validacao", preds_val)):
    pred = df[split][colunas].copy()
    pred["p_cnn"], pred["p_resnet"] = preds["CNN do zero"], preds["ResNet50 (TL)"]
    pred.to_csv(MODEL_DIR / f"predicoes_{split}.csv", index=False)
print({k: f"{v:.1f} MB" for k, v in tam.items()})

## 9. Protótipo interativo — Assistente Cardiológico Virtual

A célula abaixo é a versão "notebook" do protótipo. Envie uma radiografia de tórax (PNG ou JPG) e o assistente mostra:

* a **classe prevista** e a probabilidade de cardiomegalia;
* a **faixa de confiança** em linguagem simples;
* o **Grad-CAM** sobreposto à imagem;
* um **aviso** de que o resultado é apoio à decisão e não substitui o laudo médico.

O mesmo fluxo está disponível como **interface web em Flask** (`src/api`) e como **app React Native** (`mobile/`), descritos no README.

In [ ]:
from PIL import Image

def assistente(imagem_pil, nome_modelo="ResNet50 (TL)"):
    img = np.asarray(imagem_pil.convert("L").resize((TAMANHO, TAMANHO), Image.Resampling.LANCZOS), dtype=np.float32)[..., None]
    p, cam = gradcam(modelos[nome_modelo], img, CAMADA_GRADCAM[nome_modelo])
    lim = limiares[nome_modelo]
    classe = "cardiomegalia" if p >= lim else "normal"
    distancia = abs(p - lim)
    confianca = "alta" if distancia > 0.3 else "moderada" if distancia > 0.12 else "baixa — revisar com atenção"
    fig, ax = plt.subplots(1, 3, figsize=(12, 4), gridspec_kw={"width_ratios": [1, 1, 1.1]})
    ax[0].imshow(img[..., 0], cmap="gray"); ax[0].set_title("radiografia enviada"); ax[0].axis("off")
    ax[1].imshow(img[..., 0], cmap="gray"); ax[1].imshow(cam, cmap="jet", alpha=.35); ax[1].set_title("onde o modelo olhou (Grad-CAM)"); ax[1].axis("off")
    cor = "#c0392b" if classe == "cardiomegalia" else "#1e8449"
    ax[2].barh(["normal", "cardiomegalia"], [1 - p, p], color=["#1e8449", "#c0392b"]); ax[2].axvline(lim, color="k", ls="--", lw=1)
    ax[2].set_xlim(0, 1); ax[2].set_title("probabilidades (-- limiar)")
    ax[2].text(0.02, -0.75, f"Resultado: {classe.upper()}\nProbabilidade de cardiomegalia: {p:.0%}\nConfiança: {confianca}",
               color=cor, fontsize=10, fontweight="bold", va="top", transform=ax[2].get_yaxis_transform())
    fig.text(0.5, -0.06, "Ferramenta de apoio educacional — não substitui avaliação médica. Modelo: " + nome_modelo,
             ha="center", fontsize=8, color="gray")
    plt.tight_layout()
    return fig, {"classe": classe, "probabilidade_cardiomegalia": round(p, 4), "limiar": round(lim, 3), "confianca": confianca}

# Demonstração automática com dois exames do teste (um de cada classe)
for _, r in pd.concat([teste.nlargest(1, "p_tl"), teste.nsmallest(1, "p_tl")]).iterrows():
    fig, res = assistente(Image.open(DATA_DIR / r.caminho))
    fig.suptitle(f"Exame de teste {r.arquivo} — rótulo real: {r.rotulo}", y=1.03)
    fig.savefig(FIG_DIR / f"02_prototipo_{r.rotulo}.png", bbox_inches="tight"); plt.show()
    print(res)

In [ ]:
# Versão interativa (rodar no Jupyter/Colab): botão de upload + seletor de modelo
try:
    import io
    import ipywidgets as w
    from IPython.display import display, clear_output
    upload = w.FileUpload(accept=".png,.jpg,.jpeg", multiple=False, description="Enviar raio-X")
    seletor = w.ToggleButtons(options=list(modelos), value="ResNet50 (TL)", description="Modelo:")
    saida = w.Output()

    def ao_enviar(_):
        with saida:
            clear_output(wait=True)
            if not upload.value:
                return
            arq = upload.value[0] if isinstance(upload.value, (list, tuple)) else list(upload.value.values())[0]
            fig, res = assistente(Image.open(io.BytesIO(bytes(arq["content"]))), seletor.value)
            plt.show(); print(res)

    upload.observe(ao_enviar, names="value"); seletor.observe(ao_enviar, names="value")
    display(w.VBox([w.HTML("<h3>🫀 CardioIA — Assistente Cardiológico Virtual</h3>"), w.HBox([upload, seletor]), saida]))
except ImportError:
    print("Instale ipywidgets para usar o upload interativo.")

## Conclusões

Os números finais e a discussão estão na tabela da seção 6 e no README. Os pontos principais:

* O **transfer learning** começa com filtros já treinados e chega a uma AUC melhor em menos épocas. A CNN do zero, menor e mais rápida, serve como *baseline* honesto.
* Com o limiar escolhido na validação, as duas abordagens trocam precisão por **sensibilidade**. Em triagem, um falso negativo (cardiomegalia não detectada) costuma custar mais caro que um falso positivo, que segue para revisão médica.
* O Grad-CAM permite checar se o modelo olha para o coração. Os rótulos do NIH vêm de NLP aplicado aos laudos e têm ruído conhecido, então o modelo é um **protótipo educacional**, não um dispositivo médico.
* A análise de vieses por sexo, idade e incidência (PA × AP) está no notebook `03_fairness_etica.ipynb` (Ir Além 1).